In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
print("GPU disponible :", torch.cuda.is_available())
print("Device :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun GPU détecté")

GPU disponible : True
Device : Tesla T4


In [3]:
import shutil, os

shutil.rmtree("/content/xView_local", ignore_errors=True)
print("Dossier local supprimé complètement.")

Dossier local supprimé complètement.


In [4]:
drive_training_data = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/training_data"
local_training_data = "/content/xView_local/training_data"

os.makedirs(f"{local_training_data}/images/train", exist_ok=True)
os.makedirs(f"{local_training_data}/images/val", exist_ok=True)
os.makedirs(f"{local_training_data}/labels/train", exist_ok=True)
os.makedirs(f"{local_training_data}/labels/val", exist_ok=True)

for split in ["train", "val"]:
    shutil.copytree(f"{drive_training_data}/images/{split}", f"{local_training_data}/images/{split}", dirs_exist_ok=True)
    shutil.copytree(f"{drive_training_data}/labels_merged/{split}", f"{local_training_data}/labels/{split}", dirs_exist_ok=True)

print("Copie terminée.")

Copie terminée.


In [5]:
import os

drive_training_data = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/training_data"

for split in ["train", "val"]:
    n_img_drive = len(os.listdir(f"{drive_training_data}/images/{split}"))
    n_lbl_drive = len(os.listdir(f"{drive_training_data}/labels_merged/{split}"))
    print(f"[Drive] {split} : {n_img_drive} images, {n_lbl_drive} labels")

local_training_data = "/content/xView_local/training_data"
for split in ["train", "val"]:
    img_path = f"{local_training_data}/images/{split}"
    lbl_path = f"{local_training_data}/labels/{split}"
    n_img_local = len(os.listdir(img_path)) if os.path.exists(img_path) else 0
    n_lbl_local = len(os.listdir(lbl_path)) if os.path.exists(lbl_path) else 0
    print(f"[Local] {split} : {n_img_local} images, {n_lbl_local} labels")

[Drive] train : 10762 images, 10762 labels
[Drive] val : 1878 images, 1878 labels
[Local] train : 10762 images, 10762 labels
[Local] val : 1878 images, 1878 labels


In [6]:
import shutil, yaml

drive_yaml_merged_path = f"{drive_training_data}/data_merged.yaml"
local_yaml_merged_path = f"{local_training_data}/data_merged.yaml"

shutil.copy(drive_yaml_merged_path, local_yaml_merged_path)

with open(local_yaml_merged_path) as f:
    data_yaml_merged = yaml.safe_load(f)

print("nc :", data_yaml_merged["nc"])
print("names :", data_yaml_merged["names"])
print("path :", data_yaml_merged["path"])

nc : 29
names : ['Aircraft', 'Aircraft Hangar', 'Boat', 'Building', 'Bus', 'Construction Equipment', 'Construction Site', 'Container Ship', 'Damaged Building', 'Engineering Vehicle', 'Excavator', 'Facility', 'Front loader/Bulldozer', 'Helicopter', 'Helipad', 'Hut/Tent', 'Oil Tanker', 'Passenger Vehicle', 'Pylon', 'Rail Vehicle', 'Shed', 'Shipping Container', 'Shipping container lot', 'Small Car', 'Storage Tank', 'Tower', 'Trailer', 'Truck', 'Vehicle Lot']
path : /content/xView_local/training_data


In [7]:
if data_yaml_merged["path"] != local_training_data:
    data_yaml_merged["path"] = local_training_data
    with open(local_yaml_merged_path, "w") as f:
        yaml.dump(data_yaml_merged, f, sort_keys=False)
    print("Chemin corrigé et resauvegardé.")
else:
    print("Chemin déjà correct.")

Chemin déjà correct.


In [8]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.3 MB/s eta 0:00:00


In [10]:
run_name = "xview_yolov8s_merged_v2"

In [13]:
import shutil, os

drive_runs_dir = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/runs"

def backup_to_drive(trainer):
    dest = f"{drive_runs_dir}/{run_name}"
    os.makedirs(dest, exist_ok=True)
    last_path = f"{trainer.save_dir}/weights/last.pt"
    best_path = f"{trainer.save_dir}/weights/best.pt"
    try:
        if os.path.exists(last_path):
            shutil.copy(last_path, f"{dest}/last.pt")
        if os.path.exists(best_path):
            shutil.copy(best_path, f"{dest}/best.pt")
        print(f"[backup] epoch {trainer.epoch} sauvegardé sur Drive")
    except Exception as e:
        print(f"[backup] échec à l'epoch {trainer.epoch} : {e}")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
model.add_callback("on_fit_epoch_end", backup_to_drive)

results = model.train(
    data=local_yaml_merged_path,
    epochs=150,
    imgsz=640,
    batch=16,
    patience=30,
    save_period=1,
    project="/content/xView_local/runs",
    name=run_name
)

Ultralytics 8.4.160 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/xView_local/training_data/data_merged.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=xview_yolov8s_merged_v2-

In [ ]:
drive_runs_dir = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView/runs"
shutil.copytree(
    "/content/xView_local/runs/xview_yolov8s_merged",
    f"{drive_runs_dir}/xview_yolov8s_merged",
    dirs_exist_ok=True
)
print("Résultats copiés sur Drive.")

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/xView_local/runs/xview_yolov8s_merged/weights/last.pt")
results = model.train(resume=True)

In [ ]:
import os
run_dir = "/content/xView_local/runs/xview_yolov8s_merged"
print("Le dossier existe encore :", os.path.exists(run_dir))
if os.path.exists(run_dir):
    print(os.listdir(f"{run_dir}/weights"))

Le dossier existe encore : False
